# location

In [ ]:
folder = "D:\\dataset\\"
sample_size = 1500
training_per_folder = 40
validation_per_folder = 10

In [23]:
#import lib
from pathlib import Path
import json
import random
from collections import defaultdict
import shutil


BASE_DIR = Path(folder)
DATA_DIR = BASE_DIR 
val_json_path = BASE_DIR / "val.json"
train_mini_json_path = BASE_DIR / "train_mini.json"

#if output dir does not exist create one
out_dir = BASE_DIR / ("sampled_"+str(sample_size))
out_dir.mkdir(parents=True, exist_ok=True)

seed = 0

In [24]:
# read the json folder and categoary
with open(train_mini_json_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(val_json_path, "r", encoding="utf-8") as f:
    val_data = json.load(f)

print(train_data.keys())
print(val_data.keys())

print("train images:", len(train_data["images"]))
print("train annotations:", len(train_data["annotations"]))

print("val images:", len(val_data["images"]))
print("val annotations:", len(val_data["annotations"]))

dict_keys(['info', 'images', 'categories', 'annotations', 'licenses'])
dict_keys(['info', 'images', 'categories', 'annotations', 'licenses'])
train images: 500000
train annotations: 500000
val images: 100000
val annotations: 100000


In [25]:
random.seed(seed)

# get the set of which categoary
val_categories = set(ann["category_id"] for ann in val_data["annotations"])
# sort them
val_categories = sorted(val_categories)
# chose from the request number
selected_categories = random.sample(val_categories, sample_size)
selected_categories_set = set(selected_categories)

print("selected:", len(selected_categories))
print(selected_categories[:10])



selected: 20
[6311, 6890, 663, 4242, 8376, 7961, 6634, 4969, 7808, 5866]


In [26]:
def filter_json_by_categories(data, selected_categories_set):
    # filter annotations that selected
    new_annotations = [ann for ann in data["annotations"] if ann["category_id"] in selected_categories_set]
    # recird the ids
    selected_image_ids = set(ann["image_id"] for ann in new_annotations)

    # filter images that needs to copy
    new_images = [img for img in data["images"] if img["id"] in selected_image_ids ]

    # filter catgoary
    new_data = {}
    # info and licenses will not be modified
    for key, value in data.items():
        if key not in ["images", "annotations", "categories"]:
            new_data[key] = value
    # change to the selected data
    new_data["images"] = new_images
    new_data["annotations"] = new_annotations
    # update categories
    if "categories" in data:
        new_data["categories"] = [ cat for cat in data["categories"] if cat["id"] in selected_categories_set]

    return new_data

In [27]:
def get_image_path(img):
    return img.get("file_name") or img.get("path") or img.get("filepath")


def subset_json_by_image_ids(data, image_ids, output_split=None):
    # create the new json
    # selected id will be keppeed
    
    image_ids = set(image_ids)
    subset = {key: value for key, value in data.items() if key not in ["images", "annotations"]}

    subset["images"] = []
    for img in data["images"]:
        if img["id"] not in image_ids:
            continue
        # update the file path in the json file
        new_img = img.copy()
        #output split  is validation,copy somewhere else new location
        # if just train nothing happen
        if output_split is not None:
            path_key = next((key for key in ["file_name", "path", "filepath"] if new_img.get(key)),None,)
            old_path = Path(new_img[path_key])
            new_img[path_key] = (Path(output_split) / Path(*old_path.parts[1:])).as_posix()

        subset["images"].append(new_img)

    subset["annotations"] = [ ann for ann in data["annotations"] if ann["image_id"] in image_ids]
    return subset

In [28]:


def split_each_folder(data, training_count, validation_count, seed):
    # this perfprm the 40/10 split
    images_by_folder = defaultdict(list)
    for img in data["images"]:
        image_path = get_image_path(img)
        if image_path is None:
            raise KeyError(f"No file_name/path found in image: {img}")
        images_by_folder[Path(image_path).parent].append(img)

    # random picking
    rng = random.Random(seed)
    train_ids = set()
    validation_ids = set()
    required = training_count + validation_count

    for folder_path in sorted(images_by_folder, key=str):
        folder_images = sorted(images_by_folder[folder_path], key=lambda img: img["id"])
        if len(folder_images) < required:
            raise ValueError(
                f"{folder_path} has {len(folder_images)} images; {required} are required"
            )
        rng.shuffle(folder_images)
        train_ids.update(img["id"] for img in folder_images[:training_count])
        validation_ids.update(
            img["id"] for img in folder_images[training_count:required]
        )

    train_subset = subset_json_by_image_ids(data, train_ids, output_split="train_mini")
    validation_subset = subset_json_by_image_ids(
        data, validation_ids, output_split="validation"
    )
    return train_subset, validation_subset

# update the new json folder
selected_train_data = filter_json_by_categories(train_data, selected_categories_set)
train_500, validation_500 = split_each_folder(
    selected_train_data, training_per_folder, validation_per_folder, seed
)

# Keep the existing val filtering logic unchanged.
val_500 = filter_json_by_categories(val_data, selected_categories_set)

print("training images:", len(train_500["images"]))
print("validation images:", len(validation_500["images"]))
print("val images:", len(val_500["images"]))

training images: 800
validation images: 200
val images: 200


In [29]:
train_out_json = out_dir / "train_mini.json"
validation_out_json = out_dir / "validation.json"
val_out_json = out_dir / "val.json"

with open(train_out_json, "w", encoding="utf-8") as f:
    json.dump(train_500, f, indent=2)

with open(validation_out_json, "w", encoding="utf-8") as f:
    json.dump(validation_500, f, indent=2)

with open(val_out_json, "w", encoding="utf-8") as f:
    json.dump(val_500, f, indent=2)

print("Saved:", train_out_json)
print("Saved:", validation_out_json)
print("Saved:", val_out_json)



Saved: D:\dataset\sampled_20\train_mini.json
Saved: D:\dataset\sampled_20\validation.json
Saved: D:\dataset\sampled_20\val.json


In [30]:
def copy_split_images(images, source_split, destination_split):
    missing = 0

    for i, img in enumerate(images, start=1):
        image_path = Path(get_image_path(img))
        relative_path = Path(*image_path.parts[1:])
        src = DATA_DIR / source_split / relative_path
        dst = out_dir / destination_split / relative_path

        if not src.exists():
            print("Missing folder:", src)
            missing += 1
            continue

        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

        if i % 50 == 0:
            print(f"Copied {i}/{len(images)} images to {destination_split}")

    print("Done.")
    print("Missing folders:", missing)


def reset_output_split(split_name):
    split_dir = out_dir / split_name
    if split_dir.exists():
        shutil.rmtree(split_dir)
    split_dir.mkdir(parents=True, exist_ok=True)


for split_name in ["train_mini", "validation", "val"]:
    reset_output_split(split_name)

# traing image copy
copy_split_images(train_500["images"], "train_mini", "train_mini")
# create the new json folder copy
copy_split_images(validation_500["images"], "train_mini", "validation")

# Keep copying the existing val split as before.
copy_split_images(val_500["images"], "val", "val")

Copied 50/800 images to train_mini
Copied 100/800 images to train_mini
Copied 150/800 images to train_mini
Copied 200/800 images to train_mini
Copied 250/800 images to train_mini
Copied 300/800 images to train_mini
Copied 350/800 images to train_mini
Copied 400/800 images to train_mini
Copied 450/800 images to train_mini
Copied 500/800 images to train_mini
Copied 550/800 images to train_mini
Copied 600/800 images to train_mini
Copied 650/800 images to train_mini
Copied 700/800 images to train_mini
Copied 750/800 images to train_mini
Copied 800/800 images to train_mini
Done.
Missing folders: 0
Copied 50/200 images to validation
Copied 100/200 images to validation
Copied 150/200 images to validation
Copied 200/200 images to validation
Done.
Missing folders: 0
Copied 50/200 images to val
Copied 100/200 images to val
Copied 150/200 images to val
Copied 200/200 images to val
Done.
Missing folders: 0
